In [1]:
from probill.agents import TeachableMcpAgent, McpHostAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import StdioServerParams, mcp_server_tools
from autogen_agentchat.ui import Console
from typing import Dict
from autogen_ext.experimental.task_centric_memory.utils import ChatCompletionClientRecorder
import json
import os
from probill.utils import create_oai_client, load_yaml_file, create_stdio_server, export_component

config_filepath = "./test.yaml"
config = load_yaml_file(config_filepath)

client = create_oai_client(config["client"])

server_params=create_stdio_server(config["StdioServerParams"])
print(server_params)
mode = "replay" #replay

# client = ChatCompletionClientRecorder(
#     client=client,
#     mode=mode,
#     session_file_path="./sessions/session.json",
# )

command='uvx' args=['mcp-obsidian'] env={'OBSIDIAN_API_KEY': '37d8b00e7e67bd386a2961c37aaf451dc1424ea5d88c37e0c71c7c93eeb04f7c'} cwd=None encoding='utf-8' encoding_error_handler='strict' read_timeout_seconds=30.0


In [6]:
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client

async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(
            read, write
        ) as session:
            # Initialize the connection
            await session.initialize()
            print("session initiated", flush=True)
            # List available prompts
            # prompts = await session.list_prompts()

            # Get a prompt
            # prompt = await session.get_prompt(
            #     "evaluation", arguments={"patient_id": "P010", "start_page_id": "BINV-20", "clinical_data": {"HER2": "positive"}}
            # )

            # List available resources
            # resources = await session.list_resources()

            # List available tools
            tools = await session.list_tools()
            for tool in tools.tools:
                print(f"Tool name:\t{tool.name}")
                print(f"Description:\t{tool.description}")
                print("-"*100)
            # # Read a resource
            # content, mime_type = await session.read_resource("file://some/path")

            # Call a tool
            # result = await session.call_tool(
            #     "evaluate_patient_guidelines", 
            #     arguments={
            #         "patient_id": "P010",
            #         "start_page_id": "BINV-20"
            #     }
            # )
            
            # print(result.content)

await run()

session initiated
Tool name:	obsidian_list_files_in_dir
Description:	Lists all files and directories that exist in a specific Obsidian directory.
----------------------------------------------------------------------------------------------------
Tool name:	obsidian_list_files_in_vault
Description:	Lists all files and directories in the root directory of your Obsidian vault.
----------------------------------------------------------------------------------------------------
Tool name:	obsidian_get_file_contents
Description:	Return the content of a single file in your vault.
----------------------------------------------------------------------------------------------------
Tool name:	obsidian_simple_search
Description:	Simple search for documents matching a specified text query across all files in the vault. 
            Use this tool when you want to do a simple text search
----------------------------------------------------------------------------------------------------
Tool name:	

In [ ]:
agent = McpHostAgent(
    name="obsidian_agent",
    model_client=client,
    description="Agent interacts with Obsidian contents",
    server_params=server_params,
    model_client_stream=True,
    system_message="You are an agent uses MCP tools to fufill the user's request. Say TERMINATE when you done.",
    reflect_on_tool_use=True,
)

In [ ]:
task = """List documents"""
# result = await Console(agent.run_stream(task=task), output_stats=True)

In [ ]:
type(client)

In [ ]:
agent_json = export_component(agent)
print(agent_json.model_dump_json(indent=2))

In [ ]:
agent_copy = McpHostAgent.load_component(agent_json)

In [ ]:
agent_json = export_component(agent_copy)
print(agent_json.model_dump_json(indent=2))

In [ ]:
from dataclasses import dataclass
from typing import Dict

from autogen_core.models import (
    LLMMessage,
)
from autogen_ext.models.openai.config import AzureOpenAIClientConfiguration
from pydantic import BaseModel


class GroupChatMessage(BaseModel):
    """Implements a sample message sent by an LLM agent"""

    body: LLMMessage


class RequestToSpeak(BaseModel):
    """Message type for agents to speak"""

    pass


@dataclass
class MessageChunk:
    message_id: str
    text: str
    author: str
    finished: bool

    def __str__(self) -> str:
        return f"{self.author}({self.message_id}): {self.text}"


# Define Host configuration model
class HostConfig(BaseModel):
    hostname: str
    port: int

    @property
    def address(self) -> str:
        return f"{self.hostname}:{self.port}"


# Define GroupChatManager configuration model
class GroupChatManagerConfig(BaseModel):
    topic_type: str
    max_rounds: int


# Define WriterAgent configuration model
class ChatAgentConfig(BaseModel):
    topic_type: str
    description: str
    system_message: str


# Define UI Agent configuration model
class UIAgentConfig(BaseModel):
    topic_type: str
    artificial_stream_delay_seconds: Dict[str, float]

    @property
    def min_delay(self) -> float:
        return self.artificial_stream_delay_seconds.get("min", 0.0)

    @property
    def max_delay(self) -> float:
        return self.artificial_stream_delay_seconds.get("max", 0.0)


# Define the overall AppConfig model
class AppConfig(BaseModel):
    host: HostConfig
    group_chat_manager: GroupChatManagerConfig
    writer_agent: ChatAgentConfig
    editor_agent: ChatAgentConfig
    ui_agent: UIAgentConfig
    client_config: AzureOpenAIClientConfiguration = None  # type: ignore[assignment] # This was required to do custom instantiation in `load_config`


In [ ]:
import asyncio
import logging
import warnings

from autogen_core import (
    TypeSubscription,
    AgentId,
)

from autogen_core.models import UserMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime
from rich.console import Console
from rich.markdown import Markdown

import logging
import os
from typing import Any, Iterable, Type

import yaml
from autogen_core import MessageSerializer, try_get_known_serializers_for_type, RoutedAgent
from autogen_ext.models.openai.config import AzureOpenAIClientConfiguration
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

def get_serializers(types: Iterable[Type[Any]]) -> list[MessageSerializer[Any]]:
    serializers = []
    for type in types:
        serializers.extend(try_get_known_serializers_for_type(type))  # type: ignore
    return serializers  # type: ignore [reportUnknownVariableType]


# TODO: This is a helper function to get rid of a lot of logs until we find exact loggers to properly set log levels ...
def set_all_log_levels(log_leve: int):
    # Iterate through all existing loggers and set their levels
    for _, logger in logging.root.manager.loggerDict.items():
        if isinstance(logger, logging.Logger):  # Ensure it's actually a Logger object
            logger.setLevel(log_leve)  # Adjust to DEBUG or another level as needed


async def main():
    agent_runtime = GrpcWorkerAgentRuntime(host_address="10.0.1.249:50060")
    agent_runtime.add_message_serializer(get_serializers([UserMessage]))  # type: ignore[arg-type]
    Console().print(Markdown("Starting **`Agent`**"))
    await agent_runtime.start()
    agent_type = await agent.register(
        agent_runtime,
        "obsidian",
        agent,
    )
    await agent_runtime.add_subscription(
        TypeSubscription(topic_type="obsidian", agent_type="obsidian")
    )

    # Create AgentId using the agent's ID instead of trying to access type.key
    recipient = agent.id
    
    message = UserMessage(
        content=task,
        source="user"
    )

    await agent_runtime.send_message(message, recipient=recipient)
    await agent_runtime.stop_when_signal()


set_all_log_levels(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning, message="Resolved model mismatch.*")
await main()

In [ ]:
import asyncio
import logging
from dataclasses import dataclass

from autogen_core import (
    AgentId,
    DefaultSubscription,
    DefaultTopicId,
    MessageContext,
    RoutedAgent,
    message_handler,
)
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime


@dataclass
class AskToGreet:
    content: str


@dataclass
class Greeting:
    content: str


@dataclass
class Feedback:
    content: str


class ReceiveAgent(RoutedAgent):
    def __init__(self) -> None:
        super().__init__("Receive Agent")

    @message_handler
    async def on_greet(self, message: Greeting, ctx: MessageContext) -> Greeting:
        return Greeting(content=f"Received: {message.content}")

    @message_handler
    async def on_feedback(self, message: Feedback, ctx: MessageContext) -> None:
        print(f"Feedback received: {message.content}")


class GreeterAgent(RoutedAgent):
    def __init__(self, receive_agent_type: str) -> None:
        super().__init__("Greeter Agent")
        self._receive_agent_id = AgentId(receive_agent_type, self.id.key)

    @message_handler
    async def on_ask(self, message: AskToGreet, ctx: MessageContext) -> None:
        response = await self.send_message(Greeting(f"Hello, {message.content}!"), recipient=self._receive_agent_id)
        await self.publish_message(Feedback(f"Feedback: {response.content}"), topic_id=DefaultTopicId())


async def main() -> None:
    runtime = GrpcWorkerAgentRuntime(host_address="10.0.1.249:50060")
    await runtime.start()

    await ReceiveAgent.register(
        runtime,
        "receiver",
        lambda: ReceiveAgent(),
    )
    await runtime.add_subscription(DefaultSubscription(agent_type="receiver"))
    await GreeterAgent.register(
        runtime,
        "greeter",
        lambda: GreeterAgent("receiver"),
    )
    await runtime.add_subscription(DefaultSubscription(agent_type="greeter"))
    await runtime.publish_message(AskToGreet("Hello World!"), topic_id=DefaultTopicId())

    await runtime.stop_when_signal()



logging.basicConfig(level=logging.DEBUG)
logger = logging.getLogger("autogen_core")
logger.setLevel(logging.DEBUG)
await main()
